In [6]:
import ase
import numpy as np

In [ ]:
# ---------- USER DEFINED PARAMETERS HERE ----------#
trajectory_file = "example_Li.lammpstrj" #trajectory file path relative to current directory
carve_radius = 5.0 # carve radius in Angstroms  
solute_atoms = [
    [0, "S"],
    [1, "S"],
    [2, "S"],
    [3, "S"],
    [4, "S"],
    [5, "S"],
    [6, "S"],
    [7, "S"],
    [8, "Li"],
    [9, "Li"],
] # list the solute atoms as they appear in the dump file [atom_index, atom_symbol]
solvent_atoms = ['C', 'O', 'H', 'H', 'H', 'C', 'C', 'H', 'H', 'H', 'O', 'H', 'C', 'H', 'H', 'H'] # list the solvent atoms as they appear in order (ASSUMPTION: solvent order doesn't vary)
output_directory = "./carved_outputs/"

In [4]:
solute_indices = [atom[0] for atom in solute_atoms]
n_solvent_atoms = len(solvent_atoms)
n_solute_atoms = len(solute_indices)
solute_indices = set(solute_indices)
first_solute_index = min(solute_indices)
def which_solvents(frame):
    solute_list = [atom[0] for atom in solute_atoms]
    solvent_mask = np.ones(len(frame), dtype=bool)
    solvent_mask[solute_list] = False
    solvent_all = np.flatnonzero(solvent_mask)
    L = frame.cell.lengths()
    raw = frame.positions[solvent_all][None, :, :] - frame.positions[solute_list][:, None, :]
    vectors = raw - L * np.round(raw / L)
    distances = np.linalg.norm(vectors, axis=-1)
    si, sj = np.nonzero(distances < carve_radius)
    if len(si) == 0:
        return {}
    all_solute_indices = np.array(solute_list)[si]
    all_solvent_indices = solvent_all[sj]
    all_distances = distances[si, sj]
    all_vectors = vectors[si, sj]
    all_molecule_indices = np.where(
            all_solvent_indices < first_solute_index,
            all_solvent_indices // n_solvent_atoms,
            (all_solvent_indices - first_solute_index - n_solute_atoms)
            // n_solvent_atoms + first_solute_index // n_solvent_atoms
        )
    unique_molecules, inverse = np.unique(all_molecule_indices, return_inverse=True)
    minimum_distances = np.full(len(unique_molecules), np.inf)
    np.minimum.at(minimum_distances, inverse, all_distances)
    is_minimum = (all_distances == minimum_distances[inverse])
    candidate_positions = np.flatnonzero(is_minimum)
    minimum_positions = np.full(len(unique_molecules), len(all_distances), dtype=int)
    np.minimum.at(minimum_positions, inverse[candidate_positions], candidate_positions)
    dist_solv_dict = {}
    for k, molecule in enumerate(unique_molecules):
        p = minimum_positions[k]
        dist_solv_dict[int(molecule)] = {"min_distance": float(minimum_distances[k]), "solute_atom": int(all_solute_indices[p]), "solvent_atom": int(all_solvent_indices[p]), "vector": all_vectors[p].copy()}
    return dist_solv_dict

In [5]:
from ase.geometry import find_mic
def get_solvent_atom_indices(molecule_index):
    start = molecule_index * n_solvent_atoms
    atom_indices = np.arange(start, start + n_solvent_atoms)
    atom_indices[atom_indices >= first_solute_index] += n_solute_atoms
    return atom_indices

def make_whole(frame, atom_indices, anchor_atom):
    positions = frame.positions[atom_indices]
    anchor_position = frame.positions[anchor_atom]
    relative_positions, _ = find_mic(positions - anchor_position, frame.cell, frame.pbc)
    return anchor_position + relative_positions

In [ ]:
from ase import Atoms
def carver(frame, solvent_atoms):
    dist_solv_dict = which_solvents(frame)
    solute_atom_indices = [atom[0] for atom in solute_atoms]
    unwrapped_solute_positions = make_whole(frame, solute_atom_indices, solute_atom_indices[0])
    solute_lookup = {atom_index: position for position, atom_index in enumerate(solute_atom_indices)}
    cluster_positions = list(unwrapped_solute_positions).copy()
    cluster_symbols = [frame[i].symbol for i in solute_atom_indices]
    parent_indices = list(solute_atom_indices).copy()
    for i in dist_solv_dict:
        solvent_atom_indices = get_solvent_atom_indices(i)
        s = dist_solv_dict[i]["solute_atom"]
        j = dist_solv_dict[i]["solvent_atom"]
        solvent_positions = make_whole(frame, solvent_atom_indices, j)
        v = dist_solv_dict[i]["vector"]
        contact = unwrapped_solute_positions[solute_lookup[s]] + v
        placed = solvent_positions - frame.positions[j] + contact
        cluster_positions.extend(placed)
        cluster_symbols.extend(frame.symbols[solvent_atom_indices])
        parent_indices.extend(solvent_atom_indices)
    cop = np.mean(cluster_positions, axis=0)
    cluster_positions = cluster_positions - cop
    cluster = Atoms(symbols=cluster_symbols, positions=cluster_positions, pbc=False)
    cluster.info["timestep"] = frame.info["timestep"]
    cluster.info["n_solvent"] =len(dist_solv_dict) 
    cluster.info["centre_shift"] = cop
    cluster.arrays["atom_id"] = np.array(parent_indices) + 1
    return cluster

In [ ]:
from ase.io import iread, write
for frame in iread(trajectory_file, index=":", format="lammps-dump-text"):
    cluster = carver(frame, solvent_atoms)
    write(f"{output_directory}cluster_{frame.info['timestep']}.xyz", cluster)